# Orbit Wars — Kaggle long-run DRL/PPO com checkpoints

Este notebook é a versão para rodar **dentro do Kaggle** por várias horas, com checkpoints salvos por etapa. A estratégia não é PPO do zero. A linha usada é:

```text
pgs_holdwave / pool forte → dataset de imitação → BC entity → PPO por chunks → gate local → submissão
```

A regra operacional é simples: checkpoint salvo a cada chunk, `run_state.json` atualizado a cada avanço e bundle compactado em `/kaggle/working` antes do limite de tempo. Para retomar depois, adicione o output da versão anterior como Data Source no próximo Notebook.


## Como rodar no Kaggle

1. Crie um Notebook dentro da competição Orbit Wars.
2. Em **Add Data**, adicione o ZIP do projeto `orbit-wars-lab-oep-search-and-submit-exploration.zip`.
3. Se tiver um output anterior deste notebook, adicione esse output também em **Add Data**; o notebook vai procurar `run_state.json`, `bc_init.pt`, `ppo2p_latest.pt`, `ppo4p_latest.pt` e `best_submission.pt` automaticamente.
4. Em **Settings**, use GPU se disponível. Ligue Internet se precisar instalar pacotes durante o setup.
5. Rode com **Save Version / Run All**. Não dependa de execução interativa se quer preservar o output final.

O notebook para antes do limite configurado em `MAX_WALLTIME_HOURS`, empacota os artefatos e deixa tudo em `/kaggle/working/<RUN_NAME>_outputs`.


In [ ]:
from __future__ import annotations

# ======================
# CONFIGURAÇÃO PRINCIPAL
# ======================

RUN_NAME = "ow_drl_holdwave_long_v01"

# Use um valor abaixo do limite real mostrado na UI do Kaggle.
# Ex.: 8.5 para sessões de ~9h; 11.0 para sessões de ~12h.
MAX_WALLTIME_HOURS = 6.0
STOP_MARGIN_MINUTES = 25

# Se True, imprime comandos sem executar. Deixe False para treinar.
DRY_RUN = False

# Kaggle: "cuda" se GPU estiver ligada; cai para CPU se torch.cuda não estiver disponível.
REQUESTED_DEVICE = "cuda"

# Instalação/build. Deixe True no primeiro run.
INSTALL_REQUIREMENTS = True
BUILD_RUST_BINDING = True
FORCE_REEXTRACT_PROJECT = False

# Behavior Cloning.
RUN_BC_IF_MISSING = True
BC_SEEDS = "0-63"              # aumente para 0-127 se tiver tempo
BC_NUM_PLAYERS = 4
BC_EPISODE_STEPS = 96
BC_EPOCHS = 80
BC_BATCH_SIZE = 256
BC_LAUNCH_OVERSAMPLE = 6
BC_AUGMENT = True

# PPO long-run por chunks.
RUN_2P = True
RUN_4P = True
PPO_CHUNK_TIMESTEPS = 30_000    # checkpoint a cada chunk; reduza para 30_000 se o ambiente estiver lento
PPO_ROLLOUT_STEPS = 256
PPO_TRAIN_EPISODE_STEPS = 500
PPO_2P_MAX_CHUNKS = 24
PPO_4P_MAX_CHUNKS = 12
PPO_MIN_MINUTES_PER_CHUNK = 45  # tempo mínimo restante para iniciar novo chunk

# PPO hiperparâmetros conservadores para residual/anti-drift.
PPO_ENT_COEF = 0.003
PPO_LR = 2.5e-4
PPO_CLIP_COEF = 0.20
PPO_UPDATE_EPOCHS = 4
PPO_MINIBATCH_SIZE = 256
PPO_KL_TO_REF_COEF = 0.01
PPO_BC_ANCHOR_START = 0.06
PPO_BC_ANCHOR_END = 0.015
PPO_BC_ANCHOR_TEACHER = "pgs_holdwave"
PPO_BC_ANCHOR_MAX_QERR = 0.40

# Gate interno por chunk. Se BReP externo não estiver presente, o notebook enfraquece o gate removendo BReP.
STRICT_DRL_GATE = True
DRL_PROFILE_PER_CHUNK = "quick"
DRL_SEEDS_PER_CHUNK = 8          # múltiplo de 4
DRL_STEPS_PER_CHUNK = 256
DRL_JOBS = 1                     # use 1 com CUDA para evitar fork após CUDA

# Gate final. Rode no fim se ainda houver tempo.
RUN_FINAL_GATE = True
FINAL_GATE_PROFILE = "standard"
FINAL_GATE_SEEDS = 12            # múltiplo de 4; suba para 48/96 se sobrar tempo
FINAL_GATE_STEPS = 500
FINAL_GATE_JOBS = 2

# Oponentes. Repetição = peso no currículo.
OPPONENTS_2P = "producer,producer,oep,pgs_holdwave,pgs_bigwave,greedy,rush"
EVAL_OPPONENTS_2P = "producer,oep,pgs_holdwave,pgs_bigwave"

# Em 4p, cada item com + representa o trio de adversários nas seats restantes.
OPPONENTS_4P = "producer+oep+pgs_holdwave,producer+pgs_bigwave+greedy,oep+greedy+rush,pgs_holdwave+producer+pgs_bigwave"
EVAL_OPPONENTS_4P = "producer,oep,pgs_holdwave,pgs_bigwave"

# Exportação final.
EXPORT_SUBMISSION = True


In [ ]:
import os
import re
import sys
import json
import time
import glob
import shlex
import shutil
import tarfile
import zipfile
import hashlib
import datetime as dt
import subprocess
from pathlib import Path
from typing import Any

IS_KAGGLE = Path("/kaggle/input").exists() and Path("/kaggle/working").exists()
if not IS_KAGGLE:
    raise RuntimeError("Este notebook foi desenhado para rodar dentro do Kaggle. Execute-o em um Kaggle Notebook.")

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
PROJECT_WORK = WORKING_ROOT / "orbit_wars_project"
RUN_ROOT = WORKING_ROOT / RUN_NAME
OUT_ROOT = WORKING_ROOT / f"{RUN_NAME}_outputs"
LOG_DIR = OUT_ROOT / "logs"
CKPT_DIR = OUT_ROOT / "checkpoints"
REPORT_DIR = OUT_ROOT / "reports"
BUNDLE_DIR = OUT_ROOT / "bundles"
IMPORT_DIR = WORKING_ROOT / f"{RUN_NAME}_imported_previous"

for p in [RUN_ROOT, OUT_ROOT, LOG_DIR, CKPT_DIR, REPORT_DIR, BUNDLE_DIR, IMPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

START_TS = time.monotonic()
DEADLINE_TS = START_TS + float(MAX_WALLTIME_HOURS) * 3600.0
PY = sys.executable

STATE_PATH = OUT_ROOT / "run_state.json"

def now_iso() -> str:
    return dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


def seconds_left() -> float:
    return DEADLINE_TS - time.monotonic()


def minutes_left() -> float:
    return seconds_left() / 60.0


def time_guard(min_minutes: float, label: str = "") -> bool:
    left = minutes_left()
    ok = left >= float(min_minutes)
    print(f"[time_guard] {label} left={left:.1f} min required={float(min_minutes):.1f} -> {'OK' if ok else 'STOP'}")
    return ok


def run_cmd(cmd: list[str] | str, *, cwd: Path | str | None = None, log_name: str | None = None, check: bool = True, env: dict[str, str] | None = None) -> subprocess.CompletedProcess:
    if isinstance(cmd, str):
        shell = True
        printable = cmd
    else:
        shell = False
        printable = " ".join(shlex.quote(str(x)) for x in cmd)
    cwd = Path(cwd or Path.cwd())
    ts = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    safe = re.sub(r"[^a-zA-Z0-9_.-]+", "_", log_name or printable[:80]).strip("_") or "cmd"
    log_path = LOG_DIR / f"{ts}_{safe}.log"
    print(f"\n$ {printable}\n[cwd] {cwd}\n[log] {log_path}")
    if DRY_RUN:
        return subprocess.CompletedProcess(cmd, 0)
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)
    with open(log_path, "w", encoding="utf-8", errors="replace") as log:
        proc = subprocess.Popen(cmd, cwd=str(cwd), shell=shell, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=merged_env)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
        rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"Comando falhou com returncode={rc}: {printable}\nLog: {log_path}")
    return subprocess.CompletedProcess(cmd, rc)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")


def read_json(path: Path, default: Any = None) -> Any:
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return default

print("Kaggle input:", INPUT_ROOT)
print("Working:", WORKING_ROOT)
print("Output root:", OUT_ROOT)
print(f"Tempo restante inicial: {minutes_left():.1f} min")


In [ ]:
# =============================
# IMPORTAÇÃO DE RUNS ANTERIORES
# =============================

# Se você adicionar o output de uma versão anterior como Data Source, o notebook copia os arquivos-chave.

def extract_previous_archives() -> None:
    candidates = []
    for pat in ["*checkpoint_bundle*.tar.gz", "*checkpoint_bundle*.tgz", "*drl_bundle*.tar.gz", "*ow_drl*.tar.gz"]:
        candidates.extend(INPUT_ROOT.rglob(pat))
    seen = set()
    for arc in candidates:
        if arc in seen:
            continue
        seen.add(arc)
        try:
            dest = IMPORT_DIR / arc.stem.replace(".tar", "")
            dest.mkdir(parents=True, exist_ok=True)
            print(f"[resume] extraindo bundle anterior: {arc} -> {dest}")
            with tarfile.open(arc, "r:gz") as tf:
                tf.extractall(dest)
        except Exception as e:
            print(f"[resume] ignorando archive {arc}: {e}")


def find_resume_files(name: str) -> list[Path]:
    roots = [INPUT_ROOT, IMPORT_DIR, OUT_ROOT]
    hits: list[Path] = []
    for root in roots:
        if root.exists():
            hits.extend(root.rglob(name))
    # preferir outputs mais profundos/novos; mtime em /kaggle/input nem sempre é confiável, então ordena por tamanho e caminho.
    hits = sorted(set(hits), key=lambda p: (p.stat().st_size if p.exists() else 0, str(p)), reverse=True)
    return hits


def copy_first_resume_file(name: str, target: Path) -> Path | None:
    hits = find_resume_files(name)
    for src in hits:
        if src.resolve() == target.resolve() if target.exists() else False:
            continue
        if src.exists() and src.is_file():
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, target)
            print(f"[resume] {name}: {src} -> {target}")
            return target
    return None

extract_previous_archives()

for fname in ["bc_init.pt", "ppo2p_latest.pt", "ppo4p_latest.pt", "best_submission.pt"]:
    copy_first_resume_file(fname, CKPT_DIR / fname)

# Escolhe o melhor run_state.json anterior, se houver.
state_candidates = find_resume_files("run_state.json")
state: dict[str, Any] = {
    "run_name": RUN_NAME,
    "created_at": now_iso(),
    "updated_at": now_iso(),
    "bc_done": False,
    "ppo2p_chunks_done": 0,
    "ppo4p_chunks_done": 0,
    "latest_checkpoint": None,
    "best_submission_checkpoint": None,
    "notes": [],
}
if state_candidates:
    scored = []
    for p in state_candidates:
        try:
            s = read_json(p, {})
            score = int(s.get("ppo2p_chunks_done", 0)) + int(s.get("ppo4p_chunks_done", 0)) + (10 if s.get("bc_done") else 0)
            scored.append((score, str(p), s, p))
        except Exception:
            pass
    if scored:
        _, _, state, src = sorted(scored, reverse=True)[0]
        print(f"[resume] estado anterior carregado de {src}")

# Corrige ponteiros para arquivos copiados localmente.
if (CKPT_DIR / "bc_init.pt").exists():
    state["bc_done"] = True
    state["bc_checkpoint"] = str(CKPT_DIR / "bc_init.pt")
if (CKPT_DIR / "ppo4p_latest.pt").exists():
    state["latest_checkpoint"] = str(CKPT_DIR / "ppo4p_latest.pt")
elif (CKPT_DIR / "ppo2p_latest.pt").exists():
    state["latest_checkpoint"] = str(CKPT_DIR / "ppo2p_latest.pt")
elif (CKPT_DIR / "bc_init.pt").exists():
    state["latest_checkpoint"] = str(CKPT_DIR / "bc_init.pt")
if (CKPT_DIR / "best_submission.pt").exists():
    state["best_submission_checkpoint"] = str(CKPT_DIR / "best_submission.pt")

write_json(STATE_PATH, state)
print(json.dumps(state, indent=2))


In [ ]:
# =======================
# LOCALIZAR O PROJETO ZIP
# =======================

def zip_contains_project(zpath: Path) -> bool:
    try:
        with zipfile.ZipFile(zpath) as zf:
            names = zf.namelist()
            has_ppo = any(name.endswith("scripts/ppo_campaign.py") for name in names)
            has_cargo = any(name.endswith("Cargo.toml") for name in names)
            has_req = any(name.endswith("requirements.txt") for name in names)
            return bool(has_ppo and has_cargo and has_req)
    except Exception:
        return False


def find_project_source() -> tuple[str, Path]:
    # 1) Projeto já descompactado em /kaggle/input.
    for p in INPUT_ROOT.rglob("scripts/ppo_campaign.py"):
        root = p.parents[1]
        if (root / "Cargo.toml").exists() and (root / "requirements.txt").exists():
            return "dir", root
    # 2) ZIP do projeto.
    zips = sorted(INPUT_ROOT.rglob("*.zip"), key=lambda p: ("orbit" not in p.name.lower(), -p.stat().st_size))
    for z in zips:
        if zip_contains_project(z):
            return "zip", z
    raise FileNotFoundError("Não encontrei o ZIP/diretório do projeto em /kaggle/input. Adicione o orbit-wars-lab-oep-search-and-submit-exploration.zip como Data Source.")


def prepare_project() -> Path:
    kind, src = find_project_source()
    if PROJECT_WORK.exists() and not FORCE_REEXTRACT_PROJECT:
        candidates = list(PROJECT_WORK.rglob("scripts/ppo_campaign.py"))
        if candidates:
            root = candidates[0].parents[1]
            print(f"[project] reutilizando {root}")
            return root
    if PROJECT_WORK.exists():
        shutil.rmtree(PROJECT_WORK)
    PROJECT_WORK.mkdir(parents=True, exist_ok=True)
    if kind == "zip":
        print(f"[project] extraindo {src} -> {PROJECT_WORK}")
        with zipfile.ZipFile(src) as zf:
            zf.extractall(PROJECT_WORK)
    else:
        print(f"[project] copiando {src} -> {PROJECT_WORK}")
        shutil.copytree(src, PROJECT_WORK / src.name, dirs_exist_ok=True)
    candidates = list(PROJECT_WORK.rglob("scripts/ppo_campaign.py"))
    if not candidates:
        raise RuntimeError("Projeto extraído, mas scripts/ppo_campaign.py não foi encontrado.")
    root = candidates[0].parents[1]
    print(f"[project] ROOT = {root}")
    return root

ROOT = prepare_project()
os.chdir(ROOT)
for _p in (str(ROOT / "python"), str(ROOT)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
# subprocessos do run_cmd herdam os.environ; scripts importam tanto `orbit_wars_gym`
# (raiz) quanto `python.X`, entao o PYTHONPATH precisa de ROOT E ROOT/python.
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(ROOT), str(ROOT / "python"), os.environ.get("PYTHONPATH", "")]
).rstrip(os.pathsep)
print("cwd:", Path.cwd())
print("arquivos principais:", (ROOT / "requirements.txt").exists(), (ROOT / "Cargo.toml").exists())


In [ ]:
# =========================
# SETUP: DEPENDÊNCIAS + RUST
# =========================

if INSTALL_REQUIREMENTS:
    run_cmd([PY, "-m", "pip", "install", "-q", "-U", "pip"], cwd=ROOT, log_name="pip_upgrade")
    run_cmd([PY, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=ROOT, log_name="pip_requirements")

if BUILD_RUST_BINDING:
    # A imagem do Kaggle NÃO traz rustc/cargo. Bootstrap via rustup (exige Internet LIGADA).
    has_rust = subprocess.run("command -v cargo", shell=True, capture_output=True, text=True).returncode == 0
    if not has_rust:
        print("[rust] cargo não encontrado; instalando toolchain via rustup (precisa de Internet).")
        run_cmd(
            "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain stable",
            cwd=ROOT, log_name="rustup_install", check=True,
        )
    cargo_bin = str(Path.home() / ".cargo" / "bin")
    if cargo_bin not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = cargo_bin + os.pathsep + os.environ.get("PATH", "")
    run_cmd("rustc --version && cargo --version", cwd=ROOT, log_name="rustc_version", check=True)

    # `maturin develop` EXIGE um virtualenv/conda; o Kaggle roda o python do sistema (sem venv),
    # então `develop` falha com "Couldn't find a virtualenv". Solução: `maturin build` gera um wheel
    # e instalamos no interpretador atual com pip (mantém PY = python do sistema).
    # NB: o wheel se chama orbit_wars_lab-*.whl (dist name), mesmo que o módulo seja orbit_wars_rs.
    run_cmd([PY, "-m", "maturin", "build", "--release", "-m", "crates/orbit_wars_py/Cargo.toml"],
            cwd=ROOT, log_name="maturin_build")
    wheels = sorted(glob.glob(str(ROOT / "target" / "wheels" / "*.whl")),
                    key=lambda p: os.path.getmtime(p), reverse=True)
    if not wheels:
        raise RuntimeError("maturin build não gerou wheel em target/wheels/ (ver log maturin_build).")
    print("[rust] wheel:", wheels[0])
    run_cmd([PY, "-m", "pip", "install", "--force-reinstall", "--no-deps", wheels[0]],
            cwd=ROOT, log_name="pip_install_wheel")
    run_cmd([PY, "-c", "import orbit_wars_rs, pathlib; print('orbit_wars_rs:', pathlib.Path(orbit_wars_rs.__file__).resolve())"],
            cwd=ROOT, log_name="verify_binding")

# Torch/device só depois de instalar requirements.
try:
    import torch
    DEVICE = "cuda" if REQUESTED_DEVICE == "cuda" and torch.cuda.is_available() else "cpu"
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
except Exception as e:
    DEVICE = "cpu"
    print("torch indisponível ou sem CUDA, usando CPU:", repr(e))

write_json(REPORT_DIR / "environment.json", {
    "device": DEVICE,
    "requested_device": REQUESTED_DEVICE,
    "python": sys.version,
    "root": str(ROOT),
    "created_at": now_iso(),
})


In [ ]:
# ==============================
# BREP OPCIONAL + PATCH SEM BREP
# ==============================

# O ZIP referencia BReP como artefato externo. Se você tiver submission_brep.tar.gz,
# adicione como Data Source; o notebook copia para o caminho hardcoded esperado.

def find_brep_tarball() -> Path | None:
    pats = ["submission_brep.tar.gz", "*brep*.tar.gz", "*BReP*.tar.gz"]
    hits: list[Path] = []
    for pat in pats:
        hits.extend(INPUT_ROOT.rglob(pat))
    hits = [p for p in hits if p.is_file() and p.suffixes[-2:] == [".tar", ".gz"]]
    if not hits:
        return None
    return sorted(hits, key=lambda p: p.stat().st_size, reverse=True)[0]

BREP_TAR = find_brep_tarball()
if BREP_TAR:
    dst = Path.home() / "projects/Kaggle/orbit-wars-lab-B/artifacts/submission_brep.tar.gz"
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BREP_TAR, dst)
    print(f"[brep] encontrado e copiado: {BREP_TAR} -> {dst}")
    BREP_AVAILABLE = True
else:
    print("[brep] BReP externo não encontrado. Vou remover BReP dos pools/gates para o notebook rodar, mas o gate fica menos exigente.")
    BREP_AVAILABLE = False


def _sub_or_fail(pattern: str, repl: str, text: str, what: str) -> str:
    # Falha ALTO se o patch não casar (evita no-op silencioso que deixa BReP no pool/gate
    # e só estoura lá na frente). Conta as substituições efetivas.
    new_text, n = re.subn(pattern, repl, text)
    if n == 0:
        raise RuntimeError(
            f"[patch_no_brep] padrão não casou para '{what}'. O código do projeto provavelmente "
            f"mudou desde que este notebook foi escrito — revise patch_no_brep contra o ZIP atual."
        )
    return new_text


def patch_no_brep() -> None:
    # 1) collect_imitation_dataset: remove BReP do pool strong/elite.
    p = ROOT / "scripts" / "collect_imitation_dataset.py"
    txt = p.read_text(encoding="utf-8")
    txt = _sub_or_fail(
        r'STRONG_EXPERT_POOL = \("producer", "pgs_holdwave", "brep", "pgs_bigwave", "oep", "rush", "greedy"\)',
        'STRONG_EXPERT_POOL = ("producer", "pgs_holdwave", "pgs_bigwave", "oep", "rush", "greedy")',
        txt, "STRONG_EXPERT_POOL")
    txt = _sub_or_fail(
        r'ELITE_EXPERT_POOL = \("producer", "pgs_holdwave", "brep", "pgs_bigwave", "oep"\)',
        'ELITE_EXPERT_POOL = ("producer", "pgs_holdwave", "pgs_bigwave", "oep")',
        txt, "ELITE_EXPERT_POOL")
    p.write_text(txt, encoding="utf-8")

    # 2) drl_promotion_gate: remove BReP das referências obrigatórias e templates 4p.
    gp = ROOT / "scripts" / "drl_promotion_gate.py"
    g = gp.read_text(encoding="utf-8")
    g = _sub_or_fail(
        r'DRL_REFERENCES: tuple\[str, \.\.\.\] = \([\s\S]*?\)\n\nDRL_REQUIRED_2P',
        'DRL_REFERENCES: tuple[str, ...] = (\n    "producer",\n    "oep",\n    "pgs_bigwave",\n    "greedy",\n    "rush",\n    "pgs_allscripts",\n)\n\nDRL_REQUIRED_2P',
        g, "DRL_REFERENCES")
    g = _sub_or_fail(
        r'DRL_REQUIRED_2P: tuple\[str, \.\.\.\] = \([\s\S]*?\)\n\nDRL_4P_TEMPLATES',
        'DRL_REQUIRED_2P: tuple[str, ...] = (\n    "producer",\n    "oep",\n    "pgs_bigwave",\n    "greedy",\n    "rush",\n    INCUMBENT,\n)\n\nDRL_4P_TEMPLATES',
        g, "DRL_REQUIRED_2P")
    g = _sub_or_fail(
        r'DRL_4P_TEMPLATES: tuple\[tuple\[str, str, str\], \.\.\.\] = \([\s\S]*?\)\n\n\n@dataclass',
        'DRL_4P_TEMPLATES: tuple[tuple[str, str, str], ...] = (\n    ("producer", "oep", INCUMBENT),\n    ("producer", "pgs_bigwave", "greedy"),\n    ("oep", "greedy", "rush"),\n    (INCUMBENT, "producer", "pgs_bigwave"),\n)\n\n\n@dataclass',
        g, "DRL_4P_TEMPLATES")
    gp.write_text(g, encoding="utf-8")

    # Sanidade: nenhuma referência a "brep" deve sobrar nos símbolos que editamos.
    assert '"brep"' not in p.read_text(encoding="utf-8").split("STRONG_EXPERT_POOL")[1].split("\n\n")[0], "brep ainda em STRONG_EXPERT_POOL"
    print("[patch] brep removido de collect_imitation_dataset + drl_promotion_gate (verificado).")

if not BREP_AVAILABLE:
    patch_no_brep()
    print("[patch] pools/gate sem BReP aplicados.")

write_json(REPORT_DIR / "brep_status.json", {"brep_available": BREP_AVAILABLE, "brep_tar": str(BREP_TAR) if BREP_TAR else None})


In [ ]:
# ==========================
# SANIDADE RÁPIDA DO PROJETO
# ==========================

# Estes testes são curtos. Se falharem, não treine.
# NB: quando rodamos SEM BReP, patch_no_brep() reescreve drl_promotion_gate.py removendo o BReP,
# o que QUEBRA por design os testes que validam o pool COM BReP. Então sem BReP omitimos
# test_drl_promotion_gate.py da sanity (o gate ainda roda de verdade nos chunks).
sanity_tests = ["tests/test_train_bc.py", "tests/test_ppo_campaign.py"]
if BREP_AVAILABLE:
    sanity_tests.append("tests/test_drl_promotion_gate.py")
else:
    print("[sanity] BReP ausente -> omitindo test_drl_promotion_gate.py (patch_no_brep o dessincroniza por design)")
run_cmd([PY, "-m", "pytest", "-q", *sanity_tests], cwd=ROOT, log_name="pytest_drl_core", check=True)
run_cmd([PY, "scripts/smoke_test.py"], cwd=ROOT, log_name="smoke_test", check=True)


In [ ]:
# =======================
# FUNÇÕES DE SNAPSHOT/BC
# =======================

state = read_json(STATE_PATH, state)


def save_state() -> None:
    state["updated_at"] = now_iso()
    state["minutes_left"] = minutes_left()
    write_json(STATE_PATH, state)
    write_json(OUT_ROOT / "run_state.json", state)


def publish_checkpoint(src: Path, name: str, state_key: str | None = None) -> Path:
    if not src.exists():
        raise FileNotFoundError(src)
    dst = CKPT_DIR / name
    shutil.copy2(src, dst)
    meta = {
        "source": str(src),
        "published": str(dst),
        "sha256": sha256_file(dst),
        "bytes": dst.stat().st_size,
        "updated_at": now_iso(),
    }
    write_json(CKPT_DIR / f"{name}.json", meta)
    if state_key:
        state[state_key] = str(dst)
    state["latest_checkpoint"] = str(dst)
    save_state()
    print(f"[checkpoint] {src} -> {dst}")
    return dst


def bundle_outputs(tag: str) -> Path:
    save_state()
    bundle = BUNDLE_DIR / f"checkpoint_bundle_{RUN_NAME}_{tag}.tar.gz"
    include = [
        OUT_ROOT / "run_state.json",
        CKPT_DIR,
        REPORT_DIR,
        LOG_DIR,
    ]
    with tarfile.open(bundle, "w:gz") as tf:
        for item in include:
            if not item.exists():
                continue
            arcname = item.relative_to(OUT_ROOT.parent)
            tf.add(item, arcname=str(arcname))
    print(f"[bundle] {bundle} ({bundle.stat().st_size/1e6:.2f} MB)")
    return bundle


def copy_report(src: Path, name: str) -> None:
    if src.exists():
        dst = REPORT_DIR / name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print(f"[report] {src} -> {dst}")


def train_bc_if_needed() -> Path:
    existing = CKPT_DIR / "bc_init.pt"
    if existing.exists():
        print(f"[bc] usando checkpoint existente: {existing}")
        state["bc_done"] = True
        state["bc_checkpoint"] = str(existing)
        state["latest_checkpoint"] = str(existing)
        save_state()
        return existing
    if not RUN_BC_IF_MISSING:
        raise FileNotFoundError("bc_init.pt ausente e RUN_BC_IF_MISSING=False")
    if not time_guard(90, "BC completo"):
        raise TimeoutError("Sem tempo suficiente para coletar dataset + treinar BC")

    ds_out = RUN_ROOT / "imitation"
    datasets = "league_strong_mix,hard_states"
    collect_cmd = [
        PY, "scripts/collect_imitation_dataset.py",
        "--datasets", datasets,
        "--seeds", BC_SEEDS,
        "--num-players", str(BC_NUM_PLAYERS),
        "--episode-steps", str(BC_EPISODE_STEPS),
        "--enable-comets",
        "--launch-oversample", str(BC_LAUNCH_OVERSAMPLE),
        "--out-dir", str(ds_out),
        "--self-check",
    ]
    if BC_AUGMENT:
        collect_cmd.append("--augment")
    run_cmd(collect_cmd, cwd=ROOT, log_name="collect_imitation")

    ds_path = ds_out / "league_strong_mix.npz"
    if not ds_path.exists():
        raise FileNotFoundError(f"Dataset esperado não foi gerado: {ds_path}")
    copy_report(ds_out / "dataset_report.json", "bc_dataset_report.json")

    bc_out = RUN_ROOT / "bc" / "bc_entity.pt"
    run_cmd([
        PY, "-m", "python.train.train_bc",
        "--dataset", str(ds_path),
        "--arch", "entity",
        "--epochs", str(BC_EPOCHS),
        "--batch-size", str(BC_BATCH_SIZE),
        "--device", DEVICE,
        "--checkpoint-out", str(bc_out),
    ], cwd=ROOT, log_name="train_bc_entity")

    published = publish_checkpoint(bc_out, "bc_init.pt", "bc_checkpoint")
    state["bc_done"] = True
    save_state()
    bundle_outputs("after_bc")
    return published

BC_CKPT = train_bc_if_needed()
print("BC_CKPT:", BC_CKPT)


In [ ]:
# ============================
# LOOP PPO COM CHECKPOINTS 2P/4P
# ============================

def parse_campaign_report(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    try:
        return read_json(path, {})
    except Exception as e:
        print(f"[warn] não consegui ler {path}: {e}")
        return {}


def choose_stage_output(out_dir: Path) -> tuple[Path, dict[str, Any]]:
    report = parse_campaign_report(out_dir / "campaign_report.json")
    best = report.get("best_checkpoint")
    observed = report.get("best_observed_checkpoint")
    candidates = []
    for x in [best, observed, str(out_dir / "chunk00.pt")]:
        if x:
            p = Path(x)
            if p.exists():
                candidates.append(p)
    if not candidates:
        hits = sorted(out_dir.rglob("*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
        candidates.extend(hits)
    if not candidates:
        raise FileNotFoundError(f"Nenhum checkpoint gerado em {out_dir}")
    return candidates[0], report


def report_gate_pass(report: dict[str, Any]) -> bool:
    hist = report.get("history") or []
    if not hist:
        return False
    verdict = str(hist[-1].get("gate_verdict", ""))
    return verdict == "PASS_LOCAL"


def run_ppo_chunk(*, stage_name: str, chunk_idx: int, init_ckpt: Path, training_track: str, opponents: str, eval_opponents: str, num_players: int | None) -> tuple[Path, bool, dict[str, Any]]:
    out_dir = RUN_ROOT / "ppo" / stage_name / f"chunk_{chunk_idx:03d}"
    cmd = [
        PY, "scripts/ppo_campaign.py",
        "--init", str(init_ckpt),
        "--out-dir", str(out_dir),
        "--chunks", "1",
        "--chunk-timesteps", str(PPO_CHUNK_TIMESTEPS),
        "--train-episode-steps", str(PPO_TRAIN_EPISODE_STEPS),
        "--rollout-steps", str(PPO_ROLLOUT_STEPS),
        "--rollout-num-envs", "1",
        "--opponents", opponents,
        "--eval-opponents", eval_opponents,
        "--eval-seeds", str(max(4, DRL_SEEDS_PER_CHUNK)),
        "--eval-episode-steps", str(DRL_STEPS_PER_CHUNK),
        "--eval-jobs", "1",
        "--ent-coef", str(PPO_ENT_COEF),
        "--learning-rate", str(PPO_LR),
        "--clip-coef", str(PPO_CLIP_COEF),
        "--update-epochs", str(PPO_UPDATE_EPOCHS),
        "--minibatch-size", str(PPO_MINIBATCH_SIZE),
        "--device", DEVICE,
        "--training-track", training_track,
        "--kl-to-ref-coef", str(PPO_KL_TO_REF_COEF),
        "--ref-checkpoint", str(BC_CKPT),
        "--bc-anchor-coef", str(PPO_BC_ANCHOR_START),
        "--bc-anchor-coef-end", str(PPO_BC_ANCHOR_END),
        "--bc-anchor-teacher", PPO_BC_ANCHOR_TEACHER,
        "--bc-anchor-max-quant-error", str(PPO_BC_ANCHOR_MAX_QERR),
        "--patience", "99",
        "--drl-profile", DRL_PROFILE_PER_CHUNK,
        "--drl-seeds", str(DRL_SEEDS_PER_CHUNK),
        "--drl-steps", str(DRL_STEPS_PER_CHUNK),
        "--drl-jobs", str(DRL_JOBS),
        "--min-decoder-max-moves-per-turn", "2",
    ]
    if num_players is not None:
        cmd += ["--num-players", str(num_players)]
    if STRICT_DRL_GATE:
        cmd.append("--strict-drl-gate")
    run_cmd(cmd, cwd=ROOT, log_name=f"ppo_{stage_name}_{chunk_idx:03d}")
    ckpt, report = choose_stage_output(out_dir)
    passed = report_gate_pass(report)
    copy_report(out_dir / "campaign_report.json", f"ppo_{stage_name}_chunk_{chunk_idx:03d}_campaign_report.json")
    if (out_dir / "drl_gate").exists():
        # Copia relatórios json pequenos do gate.
        dst_gate = REPORT_DIR / f"ppo_{stage_name}_chunk_{chunk_idx:03d}_gate"
        if dst_gate.exists():
            shutil.rmtree(dst_gate)
        shutil.copytree(out_dir / "drl_gate", dst_gate, dirs_exist_ok=True)
    return ckpt, passed, report


def run_stage(stage_name: str, *, max_chunks: int, start_ckpt: Path, training_track: str, opponents: str, eval_opponents: str, num_players: int | None, latest_name: str, chunks_done_key: str) -> Path:
    current = start_ckpt
    done = int(state.get(chunks_done_key, 0))
    print(f"[stage] {stage_name}: chunks já feitos={done}, max={max_chunks}, start={current}")
    while done < int(max_chunks):
        if not time_guard(max(PPO_MIN_MINUTES_PER_CHUNK, STOP_MARGIN_MINUTES + 5), f"antes de {stage_name} chunk {done}"):
            print(f"[stage] sem tempo seguro para novo chunk {stage_name}. Parando.")
            break
        ckpt, passed, report = run_ppo_chunk(
            stage_name=stage_name,
            chunk_idx=done,
            init_ckpt=current,
            training_track=training_track,
            opponents=opponents,
            eval_opponents=eval_opponents,
            num_players=num_players,
        )
        current = publish_checkpoint(ckpt, latest_name, "latest_checkpoint")
        done += 1
        state[chunks_done_key] = done
        state[f"{stage_name}_last_passed_gate"] = bool(passed)
        state[f"{stage_name}_last_report"] = str(REPORT_DIR / f"ppo_{stage_name}_chunk_{done-1:03d}_campaign_report.json")
        # Só promove como best_submission se passou no gate. Se BReP não existe, isso é um gate mais fraco.
        if passed:
            publish_checkpoint(ckpt, "best_submission.pt", "best_submission_checkpoint")
            state["best_submission_source_stage"] = stage_name
            state["best_submission_source_chunk"] = done - 1
        save_state()
        bundle_outputs(f"after_{stage_name}_{done:03d}")
    return current

start_ckpt = Path(state.get("latest_checkpoint") or BC_CKPT)
if RUN_2P:
    start_ckpt = run_stage(
        "2p",
        max_chunks=PPO_2P_MAX_CHUNKS,
        start_ckpt=start_ckpt,
        training_track="phase0_2p",
        opponents=OPPONENTS_2P,
        eval_opponents=EVAL_OPPONENTS_2P,
        num_players=2,
        latest_name="ppo2p_latest.pt",
        chunks_done_key="ppo2p_chunks_done",
    )

if RUN_4P and time_guard(max(PPO_MIN_MINUTES_PER_CHUNK, STOP_MARGIN_MINUTES + 5), "antes do estágio 4p"):
    # 4p ESPECIALISTA: começa do BC (não do checkpoint 2p) — currículo misto 2p->4p
    # regrediu vs 4p-only (DB 231). Resume honrado se já houver ppo4p_latest.
    start4 = CKPT_DIR / "ppo4p_latest.pt" if (CKPT_DIR / "ppo4p_latest.pt").exists() else BC_CKPT
    run_stage(
        "4p",
        max_chunks=PPO_4P_MAX_CHUNKS,
        start_ckpt=start4,
        training_track="phase5_4p",
        opponents=OPPONENTS_4P,
        eval_opponents=EVAL_OPPONENTS_4P,
        num_players=4,
        latest_name="ppo4p_latest.pt",
        chunks_done_key="ppo4p_chunks_done",
    )

save_state()
bundle_outputs("after_ppo_loop")
print(json.dumps(read_json(STATE_PATH), indent=2))


In [ ]:
# ==================
# GATE FINAL + AUDIT
# ==================

state = read_json(STATE_PATH, state)
final_ckpt = None
for key in ["best_submission_checkpoint", "latest_checkpoint", "bc_checkpoint"]:
    val = state.get(key)
    if val and Path(val).exists():
        final_ckpt = Path(val)
        print(f"[final] usando {key}: {final_ckpt}")
        break
if final_ckpt is None:
    raise FileNotFoundError("Nenhum checkpoint final encontrado.")

if RUN_FINAL_GATE and time_guard(60, "gate final"):
    gate_dir = RUN_ROOT / "final_gate"
    # Em CUDA, força jobs=1: o gate carrega a policy em GPU e jobs>1 forka workers
    # após a inicialização do CUDA -> deadlock conhecido (ver wsl_cuda_multiprocessing).
    final_gate_jobs = 1 if DEVICE == "cuda" else FINAL_GATE_JOBS
    if final_gate_jobs != FINAL_GATE_JOBS:
        print(f"[final] DEVICE=cuda -> rebaixando jobs do gate final {FINAL_GATE_JOBS} -> {final_gate_jobs} (evita fork após CUDA)")
    run_cmd([
        PY, "scripts/drl_promotion_gate.py",
        "--checkpoint", str(final_ckpt),
        "--out-dir", str(gate_dir),
        "--profile", FINAL_GATE_PROFILE,
        "--seeds", str(FINAL_GATE_SEEDS),
        "--steps", str(FINAL_GATE_STEPS),
        "--jobs", str(final_gate_jobs),
    ], cwd=ROOT, log_name="final_drl_gate", check=False)
    copy_report(gate_dir / "report.json", "final_gate_report.json")

# Auditoria de checkpoints acumulados.
ckpt_glob = str(CKPT_DIR / "*.pt")
audit_dir = RUN_ROOT / "audit_checkpoints"
if time_guard(20, "auditoria curta"):
    run_cmd([PY, "scripts/audit_ppo_checkpoints.py", ckpt_glob, "--out-dir", str(audit_dir)], cwd=ROOT, log_name="audit_checkpoints", check=False)
    if audit_dir.exists():
        dst = REPORT_DIR / "audit_checkpoints"
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(audit_dir, dst, dirs_exist_ok=True)

save_state()
bundle_outputs("after_final_gate")


In [ ]:
# =====================
# EXPORTAR SUBMISSION.PY
# =====================

state = read_json(STATE_PATH, state)
# Bots ESPECIALISTAS roteados por num_players: modelo 2p p/ jogos 2p, modelo 4p p/ 4p.
ckpt_2p = CKPT_DIR / "ppo2p_latest.pt"
ckpt_4p = CKPT_DIR / "ppo4p_latest.pt"
bc_ck = CKPT_DIR / "bc_init.pt"
primary = ckpt_2p if ckpt_2p.exists() else (bc_ck if bc_ck.exists() else None)
if primary is None:
    raise FileNotFoundError("Sem checkpoint 2p/BC para exportação.")
print(f"[export] 2p/primary={primary}  4p={str(ckpt_4p) if ckpt_4p.exists() else 'NENHUM -> usa primary em 4p'}")

if EXPORT_SUBMISSION:
    sub_py = OUT_ROOT / "submission.py"
    export_cmd = [PY, "scripts/export_submission.py", "--checkpoint", str(primary), "--out", str(sub_py)]
    if ckpt_4p.exists():
        export_cmd += ["--checkpoint-4p", str(ckpt_4p), "--four-player-policy", "neural"]
    run_cmd(export_cmd, cwd=ROOT, log_name="export_submission")
    export_ckpt = primary
    tar_path = OUT_ROOT / "submission_ppo_kaggle.tar.gz"
    with tarfile.open(tar_path, "w:gz") as tf:
        tf.add(sub_py, arcname="submission.py")
    print(f"[export] {sub_py}")
    print(f"[export] {tar_path}")
    state["export_checkpoint"] = str(export_ckpt)
    state["submission_py"] = str(sub_py)
    state["submission_tar"] = str(tar_path)
    save_state()

final_bundle = bundle_outputs("FINAL")
print("\nARTEFATOS FINAIS:")
for p in [OUT_ROOT / "submission.py", OUT_ROOT / "submission_ppo_kaggle.tar.gz", final_bundle, OUT_ROOT / "run_state.json"]:
    print(" -", p, "exists=", p.exists())


In [ ]:
# =========================
# LINKS DE DOWNLOAD NO KAGGLE
# =========================

from IPython.display import FileLink, display

print("Use a aba Output do Kaggle para baixar estes arquivos depois do Save Version.")
for p in [OUT_ROOT / "submission_ppo_kaggle.tar.gz", OUT_ROOT / "submission.py", OUT_ROOT / "run_state.json"]:
    if p.exists():
        display(FileLink(str(p)))
for p in sorted(BUNDLE_DIR.glob("checkpoint_bundle_*.tar.gz"))[-3:]:
    display(FileLink(str(p)))


## Retomada depois do primeiro long-run

No próximo Notebook Kaggle:

1. Abra esta mesma versão do notebook ou faça fork.
2. Em **Add Data**, adicione o output da versão anterior.
3. Rode tudo de novo.

O notebook vai detectar os arquivos `bc_init.pt`, `ppo2p_latest.pt`, `ppo4p_latest.pt`, `best_submission.pt` e `run_state.json`. Se o primeiro run parou depois de 8 chunks 2p, ele continua a partir desse ponto. Se já entrou em 4p, continua do `ppo4p_latest.pt`.

Se quiser forçar uma nova campanha, troque `RUN_NAME` ou apague o output anterior dos Data Sources.
